# **DATA WAREHOUSE BERITA HOAKS INDONESIA**



## **Analisis dan Integrasi Data Berita Hoaks Multi Sumber**


Notebook ini disusun untuk membangun proses ETL (*Extract, Transform, Load*) pada data berita hoaks dari berbagai sumber berita digital di Indonesia.

Struktur notebook dirancang mengikuti tahapan pembangunan data warehouse berbasis star schema agar proses integrasi, transformasi, dan analisis data menjadi lebih terstruktur.

## **Alur Pengerjaan Notebook**

1. Setup & Extract Data: Import library dan memuat dataset dari berbagai sumber berita.
2. Data Understanding: Melakukan eksplorasi struktur data, tipe data, missing value, serta format atribut.
3. Data Preprocessing & Transform: Membersihkan, menyeragamkan, dan mentransformasi data agar konsisten.
4. Integrasi Dataset: Menggabungkan seluruh dataset berita menjadi master dataset.
5. Build Dimension Tables: Membangun tabel dimensi untuk skema data warehouse.
6. Generate Fact Table: Membentuk fact table sebagai pusat analisis data.
7. Export SQL Data Warehouse: Mengekspor seluruh tabel ke format SQL untuk implementasi database.

## **Tujuan Analisis**

* Mengintegrasikan data berita hoaks dari berbagai sumber digital Indonesia.
* Membersihkan dan mentransformasi data agar memiliki format yang konsisten.
* Membangun data warehouse berbasis star schema.
* Menghasilkan tabel dimensi dan fact table untuk kebutuhan Business Intelligence.
* Mengekspor hasil ETL ke dalam format SQL agar dapat diimplementasikan pada DBMS.
* Melakukan analisis OLAP untuk menemukan pola penyebaran berita hoaks berdasarkan sumber, waktu, kategori, dan topik.

# **1. SETUP & EXTRACT**

## 1.1 Import Library

Pada tahap awal, dilakukan proses import library yang dibutuhkan yaitu pandas dan numpy.

In [ ]:
import pandas as pd
import numpy as np

## 1.2 Load Dataset

Setelah itu, seluruh dataset dari berbagai sumber berita dan hoaks dimuat ke dalam DataFrame menggunakan fungsi read_csv() dan read_excel().

In [ ]:
df_komdigi = pd.read_csv('/content/komdigi_hoaks.csv')
df_cnn = pd.read_excel('/content/dataset_cnn_10k_cleaned.xlsx')
df_kompas = pd.read_excel('/content/dataset_kompas_4k_cleaned.xlsx')
df_tempo = pd.read_excel('/content/dataset_tempo_6k_cleaned.xlsx')
df_turnback = pd.read_excel('/content/dataset_turnbackhoax_10_cleaned.xlsx')

Dataset yang digunakan berasal dari beberapa sumber, yaitu:

* KOMDIGI
* CNN
* KOMPAS
* TEMPO
* TURNBACKHOAX

Seluruh dataset kemudian disimpan ke dalam dictionary agar lebih mudah diproses secara berulang.

# **2. DATA UNDERSTANDING**

Tahap ini dilakukan untuk memahami struktur dan karakteristik masing-masing dataset sebelum proses integrasi data dilakukan.

## 2.1 Pemeriksaan Struktur Dataset

Dilakukan pengecekan terhadap:

* Jumlah baris dan kolom (shape)
* Nama kolom (columns)
* Missing value (isnull)
* Contoh data (head())

Tujuan tahap ini adalah untuk mengetahui kondisi awal data dari setiap sumber.

In [ ]:
datasets = {
    "KOMDIGI": df_komdigi,
    "CNN": df_cnn,
    "KOMPAS": df_kompas,
    "TEMPO": df_tempo,
    "TURNBACKHOAX": df_turnback
}

for nama, df in datasets.items():

    print("\n================================================")
    print(f"DATASET : {nama}")
    print("================================================")

    print("\nSHAPE:")
    print(df.shape)

    print("\nKOLOM:")
    print(df.columns.tolist())

    print("\nMISSING VALUE:")
    print(df.isnull().sum())

    print("\nSAMPLE DATA:")
    display(df.head())


DATASET : KOMDIGI

SHAPE:
(16258, 13)

KOLOM:
['id', 'url', 'title', 'slug', 'published_at', 'view_count', 'excerpt', 'body_html', 'body_text', 'main_image_url', 'category', 'tags', 'topics']

MISSING VALUE:
id                    0
url                   0
title                 0
slug                  0
published_at          0
view_count            0
excerpt           15283
body_html             0
body_text             2
main_image_url       37
category              0
tags              14706
topics                0
dtype: int64

SAMPLE DATA:


,id,url,title,slug,published_at,view_count,excerpt,body_html,body_text,main_image_url,category,tags,topics
0,63885,https://www.komdigi.go.id/berita/berita-hoaks/...,[HOAKS] Vaksin TBC Mengandung Nanobots,hoaks-vaksin-tbc-mengandung-nanobots,2026-05-08 22:13:00,13,NaN,"<p style=""margin-left:0in;text-align:justify;""...",Penjelasan: Beredar sebuah unggahan di media s...,https://web.komdigi.go.id/resource/dXBsb2Fkcy8...,Klarifikasi Hoaks,hoaks hari ini,Hoaks
1,63884,https://www.komdigi.go.id/berita/berita-hoaks/...,[HOAKS] Indonesia Bakal Pungut Tarif ke Kapal ...,hoaks-indonesia-bakal-pungut-tarif-ke-kapal-as...,2026-05-08 22:10:00,13,NaN,"<p style=""margin-left:0in;text-align:justify;""...",Penjelasan: Beredar sebuah unggahan video di m...,https://web.komdigi.go.id/resource/dXBsb2Fkcy8...,Klarifikasi Hoaks,hoaks hari ini,Hoaks
2,63883,https://www.komdigi.go.id/berita/berita-hoaks/...,[HOAKS] Video Donald Trump Kibarkan Bendera Putih,hoaks-video-donald-trump-kibarkan-bendera-putih,2026-05-08 22:07:00,12,NaN,"<p style=""margin-left:0in;text-align:justify;""...",Penjelasan: Beredar sebuah unggahan video di m...,https://web.komdigi.go.id/resource/dXBsb2Fkcy8...,Klarifikasi Hoaks,hoaks hari ini,Hoaks
3,63882,https://www.komdigi.go.id/berita/berita-hoaks/...,[HOAKS] Keterlibatan Oknum TNI di Kasus Pengge...,hoaks-keterlibatan-oknum-tni-di-kasus-penggere...,2026-05-08 22:03:00,13,NaN,"<p style=""margin-left:0in;text-align:justify;""...",Penjelasan: Beredar unggahan video di media so...,https://web.komdigi.go.id/resource/dXBsb2Fkcy8...,Klarifikasi Hoaks,hoaks hari ini,Hoaks
4,63881,https://www.komdigi.go.id/berita/berita-hoaks/...,[HOAKS] Tautan untuk Daftar Seleksi Pegawai Te...,hoaks-tautan-untuk-daftar-seleksi-pegawai-teta...,2026-05-08 22:00:00,12,NaN,"<p style=""margin-left:0in;text-align:justify;""...",Penjelasan: Beredar sebuah unggahan di media s...,https://web.komdigi.go.id/resource/dXBsb2Fkcy8...,Klarifikasi Hoaks,hoaks hari ini,Hoaks



DATASET : CNN

SHAPE:
(9630, 9)

KOLOM:
['Unnamed: 0', 'Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url', 'text_new', 'hoax']

MISSING VALUE:
Unnamed: 0    0
Title         0
Timestamp     0
FullText      0
Tags          3
Author        0
Url           0
text_new      0
hoax          0
dtype: int64

SAMPLE DATA:


,Unnamed: 0,Title,Timestamp,FullText,Tags,Author,Url,text_new,hoax
0,0,Anies di Milad BKMT: Pengajian Menghasilkan Ib...,"Selasa, 21 Feb 2023 21:22 WIB","Jakarta, CNN Indonesia -- Mantan Gubernur DKI ...",anies baswedan;pengajian;pilpres 2024;badan ko...,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,Anies di Milad BKMT: Pengajian Menghasilkan Ib...,0
1,1,Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...,"Selasa, 21 Feb 2023 20:46 WIB","Medan, CNN Indonesia -- Gubernur Sumatera Utar...",edy rahmayadi;pemilu 2024;pilkada 2024,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...,0
2,2,PKB Bakal Daftarkan Menaker Ida Fauziyah Jadi ...,"Selasa, 21 Feb 2023 20:33 WIB","Jakarta, CNN Indonesia -- Partai Kebangkitan B...",ida fauziyah;pkb;pemilu 2024;pileg 2024,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,PKB Bakal Daftarkan Menaker Ida Fauziyah Jadi ...,0
3,3,Gede Pasek Doakan AHY Jadi Capres atau Cawapres,"Selasa, 21 Feb 2023 19:58 WIB","Jakarta, CNN Indonesia -- Ketua Umum Partai Ke...",gede pasek suardika;ahy;pilpres 2024;pemilu 20...,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,Gede Pasek Doakan AHY Jadi Capres atau Cawapre...,0
4,4,PKN Siapkan Jabatan Khusus Buat Anas Urbaningr...,"Selasa, 21 Feb 2023 18:56 WIB","Jakarta, CNN Indonesia -- Dewan Pimpinan Pusat...",anas urbaningrum;pkn;pemilu 2024,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,PKN Siapkan Jabatan Khusus Buat Anas Urbaningr...,0



DATASET : KOMPAS

SHAPE:
(4750, 9)

KOLOM:
['Unnamed: 0', 'Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url', 'text_new', 'hoax']

MISSING VALUE:
Unnamed: 0      0
Title          21
Timestamp      21
FullText       27
Tags          158
Author        337
Url             0
text_new       27
hoax            0
dtype: int64

SAMPLE DATA:


,Unnamed: 0,Title,Timestamp,FullText,Tags,Author,Url,text_new,hoax
0,0,"Efek Ekor Jas Pencalonan Anies, Elektabilitas ...","21 Februari 2023, 15:30 WIB",Hasil jajak pendapat yang diselenggarakan Litb...,Survei Litbang Kompas;Elektabilitas Nasdem Nai...,NaN,https://video.kompas.com/watch/258152/efek-eko...,"Efek Ekor Jas Pencalonan Anies, Elektabilitas ...",0
1,1,"Ekonomi 2024 Ditargetkan Tumbuh 5,7 Persen, pa...","Kompas.com - 21/02/2023, 14:22 WIB","JAKARTA, KOMPAS.com - Pemerintah menargetkan p...",Jakarta;Ekonomi 2024,Penulis Yohana Artha Uly | Editor Aprillia Ika,http://money.kompas.com/read/2023/02/21/142238...,"Ekonomi 2024 Ditargetkan Tumbuh 5,7 Persen, pa...",0
2,2,"Survei Litbang Kompas: PDI-P, Gerindra, dan Go...","21 Februari 2023, 09:58 WIB","PDI-Perjuangan, Partai Gerindra, dan Partai Go...",Pdip;Pdi Perjuangan;Gerindra;Golkar;Demokrat;N...,NaN,https://video.kompas.com/watch/257988/survei-l...,"Survei Litbang Kompas: PDI-P, Gerindra, dan Go...",0
3,3,"Survei Litbang ""Kompas"": Popularitas Golkar Te...","Kompas.com - 21/02/2023, 05:23 WIB","JAKARTA, KOMPAS.com - Survei Litbang Kompas Ja...",Litbang Kompas;survei kepemimpinan nasional;su...,Penulis Tatang Guritno | Editor Bagus Santosa,http://nasional.kompas.com/read/2023/02/21/052...,"Survei Litbang ""Kompas"": Popularitas Golkar Te...",0
4,4,"""Endorsement"" dan Basa-basi Politik ala Jokowi...","Kompas.com - 21/02/2023, 05:20 WIB","JAKARTA, KOMPAS.com - Presiden Joko Widodo la...",capres pemilu 2024;jokowi dukung ganjar;jokowi...,Penulis Fitria Chusna Farisa | Editor Fitria C...,http://nasional.kompas.com/read/2023/02/21/052...,"""Endorsement"" dan Basa-basi Politik ala Jokowi...",0



DATASET : TEMPO

SHAPE:
(6592, 9)

KOLOM:
['Unnamed: 0', 'Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url', 'text_new', 'hoax']

MISSING VALUE:
Unnamed: 0    0
Title         0
Timestamp     0
FullText      0
Tags          1
Author        0
Url           0
text_new      0
hoax          0
dtype: int64

SAMPLE DATA:


,Unnamed: 0,Title,Timestamp,FullText,Tags,Author,Url,text_new,hoax
0,0,Ma'ruf Amin akan Saksikan Lagi Timnas Indonesi...,"Sabtu, 1 Januari 2022 17:14 WIB","TEMPO.CO, Jakarta - Wakil Presiden Ma'ruf Amin...",Ma'ruf Amin;Piala AFF 2020;Indonesia vs Thaila...,Reporter Egi Adyatama Editor Aditya Budiman,https://nasional.tempo.co/read/1545504/maruf-a...,Ma'ruf Amin akan Saksikan Lagi Timnas Indonesi...,0
1,1,Menag Yaqut Canangkan 2022 Sebagai Tahun Toler...,"Sabtu, 1 Januari 2022 15:05 WIB","TEMPO.CO, Jakarta - Menteri Agama Yaqut Cholil...",Menag;Yaqut Cholil Qoumas;Gus Yaqut;Toleransi;...,Reporter Egi Adyatama Editor Aditya Budiman,https://nasional.tempo.co/read/1545477/menag-y...,Menag Yaqut Canangkan 2022 Sebagai Tahun Toler...,0
2,2,Jokowi Ajak Masyarakat Hadapi 2022 dengan Sema...,"Sabtu, 1 Januari 2022 12:05 WIB","TEMPO.CO, Jakarta - Presiden Joko Widodo atau ...",Jokowi;2022;Pandemi Covid-19;Resesi,Reporter Antara Editor Eko Ari Wibowo,https://nasional.tempo.co/read/1545437/jokowi-...,Jokowi Ajak Masyarakat Hadapi 2022 dengan Sema...,0
3,3,"Top Nasional: Strategi Hadapi Omicron, Lemhana...","Sabtu, 1 Januari 2022 07:28 WIB","TEMPO.CO, Jakarta - Berita yang banyak menarik...",Omicron;Lemhanas;Kemenkes;WHO;Agus Widjojo,Reporter Tempo.co Editor Eko Ari Wibowo,https://nasional.tempo.co/read/1545377/top-nas...,"Top Nasional: Strategi Hadapi Omicron, Lemhana...",0
4,4,"Mulai Tahun Ini, Menteri Tjahjo Kumolo Minta P...","Sabtu, 1 Januari 2022 07:02 WIB","TEMPO.CO, Jakarta - Menteri Pendayagunaan Apar...",Tjahjo Kumolo;Menpan RB;ASN;PNS;protokol keseh...,Reporter Friski Riana Editor Syailendra Persada,https://nasional.tempo.co/read/1545310/mulai-t...,"Mulai Tahun Ini, Menteri Tjahjo Kumolo Minta P...",0



DATASET : TURNBACKHOAX

SHAPE:
(10381, 11)

KOLOM:
['Unnamed: 0', 'Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url', 'politik', 'Narasi', 'Clean Narasi', 'hoax']

MISSING VALUE:
Unnamed: 0         0
Title              0
Timestamp          0
FullText           0
Tags               0
Author             0
Url                0
politik            0
Narasi             0
Clean Narasi    3879
hoax               0
dtype: int64

SAMPLE DATA:


,Unnamed: 0,Title,Timestamp,FullText,Tags,Author,Url,politik,Narasi,Clean Narasi,hoax
0,0,[SALAH] Anies Baswedan Dekat Dengan Aliran Krs...,"Maret 1, 2023",Hasil Periksa Fakta Gabriela Nauli Sinaga (Uni...,Fitnah;Hasut;Hoax,Pemeriksa Fakta Junior,https://turnbackhoax.id/2023/03/01/salah-anies...,1,\n“BISA DILIHAT SI ONTA YAMAN NGGAK PEDULI ITU...,BISA DILIHAT SI ONTA YAMAN NGGAK PEDULI ITU AP...,1
1,1,[SALAH] Hakim Wahyu Iman Santoso Alami Kecelak...,"Maret 1, 2023",Hasil Periksa Fakta Gabriela Nauli Sinaga (Uni...,Fitnah;Hasut;Hoax,Pemeriksa Fakta Junior,https://turnbackhoax.id/2023/03/01/salah-hakim...,0,\n“ini bener gasih?? Ya Allah gimna keadaan pa...,ini bener gasih?? Ya Allah gimna keadaan pa ha...,1
2,2,[SALAH] GAMBAR MEGAWATI DAN PUAN BERMAIN SLOT,"Februari 28, 2023",Hasil Periksa Fakta Gabriela Nauli Sinaga (Uni...,Fitnah;Hasut;Hoax,Pemeriksa Fakta Junior,https://turnbackhoax.id/2023/02/28/salah-gamba...,1,\n“Nenek lampir pemimpin partai banteng bercul...,Nenek lampir pemimpin partai banteng bercula s...,1
3,3,[SALAH] JONATHAN LATUMAHINA SEORANG NASRANI DA...,"Februari 28, 2023",Hasil Periksa Fakta Gabriela Nauli Sinaga (Uni...,Fitnah;Hasut;Hoax,Pemeriksa Fakta Junior,https://turnbackhoax.id/2023/02/28/salah-jonat...,0,\n“gerombolan kulup banyak menyusup ke ormas2 ...,gerombolan kulup banyak menyusup ke ormas2 isl...,1
4,4,[SALAH] PESAN WHATSAPP DARI BMKG YANG KABARKAN...,"Februari 28, 2023",Hasil Periksa Fakta Gabriela Nauli Sinaga (Uni...,Fitnah;Hasut;Hoax,Pemeriksa Fakta Junior,https://turnbackhoax.id/2023/02/28/salah-pesan...,1,,NaN,1


## 2.2 Pemeriksaan Tipe Data

Pada tahap ini dilakukan identifikasi tipe data setiap kolom menggunakan dtypes.

Pemeriksaan ini penting untuk memastikan apakah tipe data sudah sesuai dengan kebutuhan analisis dan proses integrasi data warehouse.

In [ ]:
for nama, df in datasets.items():

    print("\n================================================")
    print(f"TIPE DATA : {nama}")
    print("================================================")

    print(df.dtypes)


TIPE DATA : KOMDIGI
id                 int64
url               object
title             object
slug              object
published_at      object
view_count         int64
excerpt           object
body_html         object
body_text         object
main_image_url    object
category          object
tags              object
topics            object
dtype: object

TIPE DATA : CNN
Unnamed: 0     int64
Title         object
Timestamp     object
FullText      object
Tags          object
Author        object
Url           object
text_new      object
hoax           int64
dtype: object

TIPE DATA : KOMPAS
Unnamed: 0     int64
Title         object
Timestamp     object
FullText      object
Tags          object
Author        object
Url           object
text_new      object
hoax           int64
dtype: object

TIPE DATA : TEMPO
Unnamed: 0     int64
Title         object
Timestamp     object
FullText      object
Tags          object
Author        object
Url           object
text_new      object
hoax      

## 2.3 Pemeriksaan Nama Kolom Dataset

Tahap ini dilakukan untuk melihat kesesuaian struktur antar dataset, khususnya nama kolom yang berbeda antar sumber data.

Perbedaan struktur ini nantinya akan diseragamkan pada proses harmonisasi kolom.

In [ ]:
for nama, df in datasets.items():

    print("\n================================================")
    print(f"KOLOM DATASET : {nama}")
    print("================================================")

    print(f"Jumlah baris : {df.shape[0]}")
    print(f"Jumlah kolom : {df.shape[1]}")
    print("Nama-Nama Kolom")
    for col in df.columns:
        print(col)



KOLOM DATASET : KOMDIGI
Jumlah baris : 16258
Jumlah kolom : 13
Nama-Nama Kolom
id
url
title
slug
published_at
view_count
excerpt
body_html
body_text
main_image_url
category
tags
topics

KOLOM DATASET : CNN
Jumlah baris : 9630
Jumlah kolom : 9
Nama-Nama Kolom
Unnamed: 0
Title
Timestamp
FullText
Tags
Author
Url
text_new
hoax

KOLOM DATASET : KOMPAS
Jumlah baris : 4750
Jumlah kolom : 9
Nama-Nama Kolom
Unnamed: 0
Title
Timestamp
FullText
Tags
Author
Url
text_new
hoax

KOLOM DATASET : TEMPO
Jumlah baris : 6592
Jumlah kolom : 9
Nama-Nama Kolom
Unnamed: 0
Title
Timestamp
FullText
Tags
Author
Url
text_new
hoax

KOLOM DATASET : TURNBACKHOAX
Jumlah baris : 10381
Jumlah kolom : 11
Nama-Nama Kolom
Unnamed: 0
Title
Timestamp
FullText
Tags
Author
Url
politik
Narasi
Clean Narasi
hoax


## 2.4 Pemeriksaan Format Tanggal

Dilakukan identifikasi format tanggal pada setiap dataset, seperti:

* published_at
* Timestamp

Tahap ini bertujuan untuk mengetahui variasi format tanggal yang digunakan sehingga dapat diproses menjadi format datetime yang konsisten.

In [ ]:
for nama, df in datasets.items():

    print("\n================================================")
    print(f"FORMAT TANGGAL : {nama}")
    print("================================================")

    if 'published_at' in df.columns:
        print(df['published_at'].head())

    elif 'Timestamp' in df.columns:
        print(df['Timestamp'].head())

    else:
        print("Kolom tanggal tidak ditemukan")


FORMAT TANGGAL : KOMDIGI
0    2026-05-08 22:13:00
1    2026-05-08 22:10:00
2    2026-05-08 22:07:00
3    2026-05-08 22:03:00
4    2026-05-08 22:00:00
Name: published_at, dtype: object

FORMAT TANGGAL : CNN
0    Selasa, 21 Feb 2023 21:22 WIB
1    Selasa, 21 Feb 2023 20:46 WIB
2    Selasa, 21 Feb 2023 20:33 WIB
3    Selasa, 21 Feb 2023 19:58 WIB
4    Selasa, 21 Feb 2023 18:56 WIB
Name: Timestamp, dtype: object

FORMAT TANGGAL : KOMPAS
0           21 Februari 2023, 15:30 WIB
1    Kompas.com - 21/02/2023, 14:22 WIB
2           21 Februari 2023, 09:58 WIB
3    Kompas.com - 21/02/2023, 05:23 WIB
4    Kompas.com - 21/02/2023, 05:20 WIB
Name: Timestamp, dtype: object

FORMAT TANGGAL : TEMPO
0    Sabtu, 1 Januari 2022 17:14 WIB
1    Sabtu, 1 Januari 2022 15:05 WIB
2    Sabtu, 1 Januari 2022 12:05 WIB
3    Sabtu, 1 Januari 2022 07:28 WIB
4    Sabtu, 1 Januari 2022 07:02 WIB
Name: Timestamp, dtype: object

FORMAT TANGGAL : TURNBACKHOAX
0        Maret 1, 2023
1        Maret 1, 2023
2    Februari 

## 2.5 Pemeriksaan Variasi Tag

Tahap ini digunakan untuk melihat variasi isi kolom tag serta mendeteksi adanya multi-value tag dengan delimiter seperti ;.

Hasil pemeriksaan ini digunakan sebagai dasar pembuatan tabel dimensi tag.

In [ ]:
for nama, df in datasets.items():

    print("\n================================================")
    print(f"VARIASI TAG : {nama}")
    print("================================================")

    if 'tags' in df.columns:

        # cek apakah ada multi value
        multi = df[
            df['tags'].astype(str).str.contains(';', na=False)
        ]['tags']

        if len(multi) > 0:

            print("Contoh Multi Value Tag:")
            print(multi.head(5))

        else:

            print("Contoh Tag:")
            print(df['tags'].dropna().head(5))

    elif 'Tags' in df.columns:

        multi = df[
            df['Tags'].astype(str).str.contains(';', na=False)
        ]['Tags']

        if len(multi) > 0:

            print("Contoh Multi Value Tag:")
            print(multi.head(5))

        else:

            print("Contoh Tag:")
            print(df['Tags'].dropna().head(5))

    else:

        print("Kolom tag tidak tersedia")


VARIASI TAG : KOMDIGI
Contoh Tag:
0    hoaks hari ini
1    hoaks hari ini
2    hoaks hari ini
3    hoaks hari ini
4    hoaks hari ini
Name: tags, dtype: object

VARIASI TAG : CNN
Contoh Multi Value Tag:
0    anies baswedan;pengajian;pilpres 2024;badan ko...
1               edy rahmayadi;pemilu 2024;pilkada 2024
2              ida fauziyah;pkb;pemilu 2024;pileg 2024
3    gede pasek suardika;ahy;pilpres 2024;pemilu 20...
4                     anas urbaningrum;pkn;pemilu 2024
Name: Tags, dtype: object

VARIASI TAG : KOMPAS
Contoh Multi Value Tag:
0    Survei Litbang Kompas;Elektabilitas Nasdem Nai...
1                                 Jakarta;Ekonomi 2024
2    Pdip;Pdi Perjuangan;Gerindra;Golkar;Demokrat;N...
3    Litbang Kompas;survei kepemimpinan nasional;su...
4    capres pemilu 2024;jokowi dukung ganjar;jokowi...
Name: Tags, dtype: object

VARIASI TAG : TEMPO
Contoh Multi Value Tag:
0    Ma'ruf Amin;Piala AFF 2020;Indonesia vs Thaila...
1    Menag;Yaqut Cholil Qoumas;Gus Yaqut;Toleran

# **3. DATA PREPROCESSING & TRANSFORM**

## 3.1 Penambahan Kolom Sumber

Setiap dataset ditambahkan kolom sumber untuk menandai asal data berita.

Contoh sumber:
* Komdigi
* CNN
* Kompas
* Tempo
* TurnBackHoax

Kolom ini nantinya digunakan pada dimensi sumber.

In [ ]:
df_komdigi['sumber'] = 'Komdigi'
df_cnn['sumber'] = 'CNN'
df_kompas['sumber'] = 'Kompas'
df_tempo['sumber'] = 'Tempo'
df_turnback['sumber'] = 'TurnBackHoax'

print("Kolom sumber berhasil ditambahkan")

Kolom sumber berhasil ditambahkan


## 3.2 Harmonisasi Nama Kolom

Dilakukan penyamaan nama kolom antar dataset menggunakan fungsi harmonisasi_kolom().

Beberapa contoh harmonisasi:

* title → judul
* body_text → isi
* published_at → tanggal
* tags → tag

Tahap ini bertujuan agar seluruh dataset memiliki struktur yang seragam.

In [ ]:
def harmonisasi_kolom(df):

    df.columns = df.columns.str.lower()

    mapping = {

        # judul
        'title': 'judul',

        # isi berita
        'body_text': 'isi',
        'fulltext': 'isi',

        # tanggal
        'published_at': 'tanggal',
        'timestamp': 'tanggal',

        # link
        'url': 'link',

        # tag
        'tags': 'tag',

        # kategori asli
        'category': 'kategori_asli',

        # label hoaks
        'hoax': 'label_hoaks',

        # clean narasi
        'clean narasi': 'clean_narasi',

        # clean text
        'text_new': 'text_clean'
    }

    df = df.rename(columns=mapping)

    return df

In [ ]:
df_komdigi = harmonisasi_kolom(df_komdigi)

df_cnn = harmonisasi_kolom(df_cnn)

df_kompas = harmonisasi_kolom(df_kompas)

df_tempo = harmonisasi_kolom(df_tempo)

df_turnback = harmonisasi_kolom(df_turnback)

print("Harmonisasi selesai")

Harmonisasi selesai


## 3.3 Pemeriksaan Missing Value pada Tag

Dilakukan pengecekan terhadap:
* Nilai NULL
* String kosong
* Tag tidak valid

Tahap ini penting karena kolom tag akan digunakan pada dimensi tag dan fact table.

In [ ]:
for name, df in {
    "komdigi": df_komdigi,
    "cnn": df_cnn,
    "kompas": df_kompas,
    "tempo": df_tempo,
    "turnback": df_turnback
}.items():
    print(name, df['tag'].isna().sum())

komdigi 14706
cnn 3
kompas 158
tempo 1
turnback 0


In [ ]:
for name, df in {
    "komdigi": df_komdigi,
    "cnn": df_cnn,
    "kompas": df_kompas,
    "tempo": df_tempo,
    "turnback": df_turnback
}.items():

    empty = df['tag'].fillna('').str.strip().eq('').sum()
    print(name, "empty tag:", empty)

komdigi empty tag: 14706
cnn empty tag: 3
kompas empty tag: 158
tempo empty tag: 1
turnback empty tag: 0


In [ ]:
for name, df in {
    "komdigi": df_komdigi,
    "cnn": df_cnn,
    "kompas": df_kompas,
    "tempo": df_tempo,
    "turnback": df_turnback
}.items():

    bad = df['tag'].isna() | df['tag'].astype(str).str.strip().eq('')
    print(name, "invalid tag:", bad.sum())

komdigi invalid tag: 14706
cnn invalid tag: 3
kompas invalid tag: 158
tempo invalid tag: 1
turnback invalid tag: 0


## 3.4 Penghapusan Kolom Tidak Diperlukan

Kolom bawaan Excel seperti unnamed: 0 dihapus karena tidak memiliki nilai analitis.

In [ ]:
for df in [
    df_cnn,
    df_kompas,
    df_tempo,
    df_turnback
]:

    if 'unnamed: 0' in df.columns:
        df.drop(columns=['unnamed: 0'], inplace=True)

print("Kolom index excel berhasil dihapus")

Kolom index excel berhasil dihapus


## 3.5 Penyamaan Struktur Kolom

Seluruh kolom dari semua dataset digabungkan menjadi satu daftar kolom utama (all_columns).

Kemudian setiap dataframe disesuaikan agar memiliki struktur kolom yang sama menggunakan fungsi samakan_kolom().

In [ ]:
all_columns = set()

for df in [
    df_komdigi,
    df_cnn,
    df_kompas,
    df_tempo,
    df_turnback
]:

    all_columns.update(df.columns)

all_columns = sorted(all_columns)

print("Total kolom gabungan:", len(all_columns))

Total kolom gabungan: 20


In [ ]:
def samakan_kolom(df, all_columns):

    for col in all_columns:

        if col not in df.columns:
            df[col] = np.nan

    df = df[all_columns]

    return df

In [ ]:
df_komdigi = samakan_kolom(df_komdigi, all_columns)

df_cnn = samakan_kolom(df_cnn, all_columns)

df_kompas = samakan_kolom(df_kompas, all_columns)

df_tempo = samakan_kolom(df_tempo, all_columns)

df_turnback = samakan_kolom(df_turnback, all_columns)

print("Struktur kolom berhasil disamakan")

Struktur kolom berhasil disamakan


## 3.6 Cleaning Ringan Data

Dilakukan pembersihan sederhana terhadap data bertipe object, seperti:

Menghapus spasi berlebih (strip)
Mengubah string "nan", "None", dan "null" menjadi NULL

Tahap ini membantu meningkatkan konsistensi data.

In [ ]:
def cleaning_ringan(df):

    for col in df.columns:

        if df[col].dtype == 'object':

            df[col] = df[col].astype(str)

            # trim
            df[col] = df[col].str.strip()

            # replace null string
            df[col] = df[col].replace(
                ['nan', 'None', 'null'],
                np.nan
            )

    return df

In [ ]:
df_komdigi = cleaning_ringan(df_komdigi)

df_cnn = cleaning_ringan(df_cnn)

df_kompas = cleaning_ringan(df_kompas)

df_tempo = cleaning_ringan(df_tempo)

df_turnback = cleaning_ringan(df_turnback)

print("Cleaning ringan selesai")

Cleaning ringan selesai


## 3.7 Parsing dan Standarisasi Tanggal

Dilakukan proses standarisasi format tanggal dari seluruh dataset agar memiliki tipe data datetime yang konsisten.

Tahap ini dilakukan karena setiap sumber dataset memiliki format tanggal yang berbeda-beda.

Proses yang dilakukan meliputi:

* Mengubah kolom tanggal menjadi string
* Menghapus teks tambahan seperti WIB, nama hari, dan prefix tertentu
* Mengubah nama bulan Indonesia menjadi bahasa Inggris
* Mengonversi tanggal ke format datetime

In [ ]:
def parsing_tanggal(df):

    if 'tanggal' in df.columns:

        # ubah ke string
        df['tanggal'] = df['tanggal'].astype(str)

        # hapus WIB
        df['tanggal'] = df['tanggal'].str.replace(
            'WIB',
            '',
            regex=False
        )

        # hapus nama hari indonesia
        hari = [
            'Senin,',
            'Selasa,',
            'Rabu,',
            'Kamis,',
            'Jumat,',
            'Sabtu,',
            'Minggu,'
        ]

        for h in hari:

            df['tanggal'] = df['tanggal'].str.replace(
                h,
                '',
                regex=False
            )

        # hapus prefix kompas
        df['tanggal'] = df['tanggal'].str.replace(
            'Kompas.com -',
            '',
            regex=False
        )

        # trim
        df['tanggal'] = df['tanggal'].str.strip()

        # mapping bulan indonesia
        bulan_map = {
            'Januari': 'January',
            'Februari': 'February',
            'Maret': 'March',
            'April': 'April',
            'Mei': 'May',
            'Juni': 'June',
            'Juli': 'July',
            'Agustus': 'August',
            'September': 'September',
            'Oktober': 'October',
            'November': 'November',
            'Desember': 'December'
        }

        for indo, eng in bulan_map.items():

            df['tanggal'] = df['tanggal'].str.replace(
                indo,
                eng,
                regex=False
            )

        # parsing datetime
        df['tanggal'] = pd.to_datetime(
            df['tanggal'],
            errors='coerce'
        )

    return df

Setelah itu, fungsi parsing diterapkan ke seluruh dataset.

Khusus dataset Kompas dilakukan backup dan reparsing tambahan karena masih terdapat beberapa format tanggal yang gagal dikonversi pada proses parsing utama.

In [ ]:
df_kompas['tanggal_backup'] = df_kompas['tanggal']

In [ ]:
df_komdigi = parsing_tanggal(df_komdigi)

df_cnn = parsing_tanggal(df_cnn)

df_kompas = parsing_tanggal(df_kompas)

df_tempo = parsing_tanggal(df_tempo)

df_turnback = parsing_tanggal(df_turnback)

In [ ]:
mask = df_kompas['tanggal'].isna()

temp = (
    df_kompas.loc[mask, 'tanggal_backup']
    .astype(str)
    .str.replace('Kompas.com -', '', regex=False)
    .str.replace('WIB', '', regex=False)
    .str.replace('|', ' ', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

# parsing format angka
parsed1 = pd.to_datetime(
    temp,
    format='%d/%m/%Y %H:%M',
    errors='coerce'
)

# parsing fleksibel
parsed2 = pd.to_datetime(
    temp,
    errors='coerce',
    dayfirst=True
)

df_kompas.loc[mask, 'tanggal'] = parsed1.fillna(parsed2)

print("Parsing tanggal selesai")

Parsing tanggal selesai


## 3.8 Pembuatan Fungsi Clean Tags

Dilakukan pembuatan fungsi clean_tags() untuk membersihkan dan menyeragamkan data tag.

* Menyeragamkan delimiter tag
* Mengubah huruf menjadi lowercase
* Menghapus tag kosong
* Menghapus duplikasi tag
* Mengisi nilai kosong dengan tidak diketahui

Tahap ini penting untuk membangun dimensi tag yang konsisten.

In [ ]:
def clean_tags(tag_str):
    if pd.isna(tag_str):
        return ['tidak diketahui']

    tag_str = str(tag_str).lower()

    # samakan semua delimiter
    tag_str = tag_str.replace(';', ',').replace('|', ',')

    tags = [t.strip() for t in tag_str.split(',')]

    # buang kosong + duplikat
    tags = list(set([t for t in tags if t != '']))

    if len(tags) == 0:
        return ['tidak diketahui']

    return tags

In [ ]:
for nama, df in {
    "komdigi": df_komdigi,
    "cnn": df_cnn,
    "kompas": df_kompas,
    "tempo": df_tempo,
    "turnback": df_turnback
}.items():

    print("\n================================================")
    print(f"VERIFIKASI CLEAN TAG : {nama}")
    print("================================================")

    all_clean_tags = []

    for tags in df['tag'].dropna():

        all_clean_tags.extend(
            clean_tags(tags)
        )

    all_clean_tags = pd.Series(all_clean_tags)

    print("Jumlah tag unik:")
    print(all_clean_tags.nunique())

    print("\nTop 10 tag:")
    print(all_clean_tags.value_counts().head(10))


VERIFIKASI CLEAN TAG : komdigi
Jumlah tag unik:
6

Top 10 tag:
hoaks hari ini    1528
perekonomian        18
kebijakan            9
komunikasi           6
umkm                 5
keamanan             3
Name: count, dtype: int64

VERIFIKASI CLEAN TAG : cnn
Jumlah tag unik:
5980

Top 10 tag:
pemilu 2024         2366
pilpres 2024        2262
jokowi              1710
pdip                1323
anies baswedan      1055
dpr                 1049
nasdem               602
prabowo subianto     556
gerindra             540
ganjar pranowo       529
Name: count, dtype: int64

VERIFIKASI CLEAN TAG : kompas
Jumlah tag unik:
8150

Top 10 tag:
politik           452
pemilu 2024       354
partai politik    354
jokowi            285
pemilu            196
pilpres 2024      186
politik uang      169
pdi-p             133
anies baswedan    130
bawaslu           118
Name: count, dtype: int64

VERIFIKASI CLEAN TAG : tempo
Jumlah tag unik:
3783

Top 10 tag:
jokowi            1885
pilpres 2024      1106
pemilu 202

# **4 INTEGRASI DATA**

## 4.1 Penggabungan Seluruh Dataset

Seluruh dataset digabungkan menggunakan pd.concat() menjadi satu master dataset bernama df_hoaks.

Tahap ini merupakan proses integrasi data utama sebelum pembangunan data warehouse.

In [ ]:
df_hoaks = pd.concat([

    df_komdigi,
    df_cnn,
    df_kompas,
    df_tempo,
    df_turnback

], ignore_index=True)

print("Semua dataset berhasil digabung")

Semua dataset berhasil digabung


## 4.2 Pembuatan Surrogate Key

Dibuat kolom id_hoaks sebagai surrogate key unik untuk setiap data berita.

Key ini digunakan sebagai identitas utama data.

In [ ]:
df_hoaks['id_hoaks'] = range(
    1,
    len(df_hoaks)+1
)

print("Surrogate key berhasil dibuat")

Surrogate key berhasil dibuat


## 4.3 Pembuatan Fitur Tambahan

Dilakukan penambahan beberapa atribut baru, yaitu:

tahun
bulan
jumlah
panjang_isi

Atribut ini digunakan untuk kebutuhan analisis dan measure pada fact table.

In [ ]:
df_hoaks['tahun'] = df_hoaks['tanggal'].dt.year

df_hoaks['bulan'] = df_hoaks['tanggal'].dt.month

df_hoaks['tahun'] = df_hoaks['tahun'].astype('Int64')

df_hoaks['bulan'] = df_hoaks['bulan'].astype('Int64')

df_hoaks['jumlah'] = 1

df_hoaks['panjang_isi'] = (
    df_hoaks['isi']
    .astype(str)
    .apply(len)
)

print("Fitur tambahan berhasil dibuat")

Fitur tambahan berhasil dibuat


## 4.4 Penghapusan Kolom Tidak Digunakan

Kolom seperti body_html dihapus karena tidak diperlukan dalam proses analisis data warehouse.

In [ ]:
if 'body_html' in df_hoaks.columns:

    df_hoaks.drop(
        columns=['body_html'],
        inplace=True
    )

print("Kolom tidak terpakai dihapus")

Kolom tidak terpakai dihapus


## 4.5 Penyesuaian Tipe Data View Count

Dilakukan konversi kolom view_count menjadi tipe data integer (Int64) agar dapat digunakan pada proses analisis numerik dan measure dalam data warehouse.

In [ ]:
df_hoaks['view_count'] = df_hoaks['view_count'].astype('Int64')

## 4.6 Verifikasi Hasil Integrasi Data

Dilakukan pemeriksaan terhadap jumlah total data dan jumlah surrogate key unik setelah proses integrasi dataset.

Tahap ini bertujuan untuk memastikan seluruh data berhasil digabung tanpa kehilangan identitas unik setiap berita.

In [ ]:
len(df_hoaks)

47611

In [ ]:
df_hoaks['id_hoaks'].nunique()

47611

In [ ]:
df_hoaks['label_hoaks'].value_counts(dropna=False)

,count
label_hoaks,
0.0,20972
NaN,16258
1.0,10381


# **5. MASTER DATA**

Dilakukan pemeriksaan akhir terhadap master dataset df_hoaks, meliputi:

* Shape dataset
* Nama kolom
* Missing value
* Sample data

Tahap ini memastikan data siap digunakan dalam pembangunan data warehouse.

In [ ]:
print("\n================================================")
print("MASTER DATA HOAKS")
print("================================================")

print("\nSHAPE:")
print(df_hoaks.shape)

print("\nKOLOM:")
print(df_hoaks.columns.tolist())

print("\nMISSING VALUE:")
print(df_hoaks.isnull().sum())

print("\nSAMPLE:")
display(df_hoaks.head())


MASTER DATA HOAKS

SHAPE:
(47611, 25)

KOLOM:
['author', 'clean_narasi', 'excerpt', 'id', 'isi', 'judul', 'kategori_asli', 'label_hoaks', 'link', 'main_image_url', 'narasi', 'politik', 'slug', 'sumber', 'tag', 'tanggal', 'text_clean', 'topics', 'view_count', 'tanggal_backup', 'id_hoaks', 'tahun', 'bulan', 'jumlah', 'panjang_isi']

MISSING VALUE:
author            16595
clean_narasi      41109
excerpt           46636
id                31353
isi                  29
judul                21
kategori_asli     31353
label_hoaks       16258
link                  0
main_image_url    31390
narasi            37230
politik           37230
slug              31353
sumber                0
tag               14868
tanggal            5218
text_clean        26666
topics            31353
view_count        31353
tanggal_backup    42882
id_hoaks              0
tahun              5218
bulan              5218
jumlah                0
panjang_isi           0
dtype: int64

SAMPLE:


,author,clean_narasi,excerpt,id,isi,judul,kategori_asli,label_hoaks,link,main_image_url,...,tanggal,text_clean,topics,view_count,tanggal_backup,id_hoaks,tahun,bulan,jumlah,panjang_isi
0,NaN,NaN,NaN,63885.0,Penjelasan: Beredar sebuah unggahan di media s...,[HOAKS] Vaksin TBC Mengandung Nanobots,Klarifikasi Hoaks,NaN,https://www.komdigi.go.id/berita/berita-hoaks/...,https://web.komdigi.go.id/resource/dXBsb2Fkcy8...,...,2026-05-08 22:13:00,NaN,Hoaks,13,NaN,1,2026,5,1,863
1,NaN,NaN,NaN,63884.0,Penjelasan: Beredar sebuah unggahan video di m...,[HOAKS] Indonesia Bakal Pungut Tarif ke Kapal ...,Klarifikasi Hoaks,NaN,https://www.komdigi.go.id/berita/berita-hoaks/...,https://web.komdigi.go.id/resource/dXBsb2Fkcy8...,...,2026-05-08 22:10:00,NaN,Hoaks,13,NaN,2,2026,5,1,1029
2,NaN,NaN,NaN,63883.0,Penjelasan: Beredar sebuah unggahan video di m...,[HOAKS] Video Donald Trump Kibarkan Bendera Putih,Klarifikasi Hoaks,NaN,https://www.komdigi.go.id/berita/berita-hoaks/...,https://web.komdigi.go.id/resource/dXBsb2Fkcy8...,...,2026-05-08 22:07:00,NaN,Hoaks,12,NaN,3,2026,5,1,701
3,NaN,NaN,NaN,63882.0,Penjelasan: Beredar unggahan video di media so...,[HOAKS] Keterlibatan Oknum TNI di Kasus Pengge...,Klarifikasi Hoaks,NaN,https://www.komdigi.go.id/berita/berita-hoaks/...,https://web.komdigi.go.id/resource/dXBsb2Fkcy8...,...,2026-05-08 22:03:00,NaN,Hoaks,13,NaN,4,2026,5,1,1533
4,NaN,NaN,NaN,63881.0,Penjelasan: Beredar sebuah unggahan di media s...,[HOAKS] Tautan untuk Daftar Seleksi Pegawai Te...,Klarifikasi Hoaks,NaN,https://www.komdigi.go.id/berita/berita-hoaks/...,https://web.komdigi.go.id/resource/dXBsb2Fkcy8...,...,2026-05-08 22:00:00,NaN,Hoaks,12,NaN,5,2026,5,1,905


# **6. BUILD DIMENSION TABLE**

Tahap ini dilakukan untuk membangun tabel dimensi pada skema star schema.

## 6.1 Pembuatan Dimensi Sumber (dim_sumber)

Dimensi ini berisi daftar sumber berita unik beserta surrogate key id_sumber.

In [ ]:
dim_sumber = (
    df_hoaks[['sumber']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_sumber['id_sumber'] = range(
    1,
    len(dim_sumber)+1
)

dim_sumber = dim_sumber[
    ['id_sumber', 'sumber']
]

print("dim_sumber selesai")

dim_sumber selesai


## 6.2 Pembuatan Dimensi Berita (dim_berita)

Dimensi berita menyimpan informasi detail berita seperti:

* Judul
* Isi
* Author
* Link
* View count

In [ ]:
dim_berita = df_hoaks[
    [
        'id_hoaks',

        'judul',

        'isi',

        'excerpt',

        'author',

        'link',

        'slug',

        'main_image_url',

        'view_count'
    ]
].copy()

dim_berita = dim_berita.rename(columns={
    'id_hoaks': 'id_berita'
})

print("dim_berita selesai")

dim_berita selesai


## 6.3 Pembuatan Dimensi Tag (dim_tag)

Dimensi tag dibuat dari seluruh tag unik yang telah dibersihkan menggunakan fungsi clean_tags().

In [ ]:
all_tags = set()

for tags in df_hoaks['tag']:
    for tag in clean_tags(tags):
        all_tags.add(tag)

dim_tag = pd.DataFrame({
    'nama_tag': sorted(all_tags)
})

dim_tag['id_tag'] = range(
    1,
    len(dim_tag)+1
)

tag_map = dict(zip(dim_tag['nama_tag'], dim_tag['id_tag']))

if 'tidak diketahui' not in tag_map:
    tag_map['tidak diketahui'] = max(tag_map.values()) + 1

print("dim_tag selesai")

dim_tag selesai


## 6.4 Pembuatan Dimensi Waktu (dim_waktu)

Dimensi waktu dibangun dari atribut tanggal dan menghasilkan atribut turunan seperti:

* Hari
* Nama hari
* Bulan
* Nama bulan
* Tahun
* Quarter

In [ ]:
dim_waktu = df_hoaks[
    ['tanggal']
].copy()

dim_waktu = dim_waktu.drop_duplicates()

dim_waktu = dim_waktu.reset_index(drop=True)

dim_waktu['id_waktu'] = range(
    1,
    len(dim_waktu)+1
)

# atribut waktu
dim_waktu['hari'] = (
    dim_waktu['tanggal']
    .dt.day
)

dim_waktu['nama_hari'] = (
    dim_waktu['tanggal']
    .dt.day_name()
)

dim_waktu['bulan'] = (
    dim_waktu['tanggal']
    .dt.month
)

dim_waktu['nama_bulan'] = (
    dim_waktu['tanggal']
    .dt.month_name()
)

dim_waktu['tahun'] = (
    dim_waktu['tanggal']
    .dt.year
)

dim_waktu['quarter'] = (
    dim_waktu['tanggal']
    .dt.quarter
)

dim_waktu = dim_waktu[
    [
        'id_waktu',
        'tanggal',
        'hari',
        'nama_hari',
        'bulan',
        'nama_bulan',
        'tahun',
        'quarter'
    ]
]

print("dim_waktu selesai")

dim_waktu selesai


## 6.5 Pembuatan Dimensi Kategori (dim_kategori)

Dimensi kategori dibentuk dari kolom kategori_asli yang telah dibersihkan.

In [ ]:
df_hoaks['kategori'] = (
    df_hoaks['kategori_asli']
    .fillna('Tidak Diketahui')
)

dim_kategori = (
    df_hoaks[['kategori']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_kategori['id_kategori'] = range(
    1,
    len(dim_kategori)+1
)

dim_kategori = dim_kategori[
    ['id_kategori', 'kategori']
]

print("dim_kategori selesai")

dim_kategori selesai


## 6.6 Pembuatan Dimensi Topik (dim_topik)

Dimensi topik dibuat dari kolom topics untuk mendukung analisis berdasarkan topik berita.

In [ ]:
df_hoaks['topics'] = (
    df_hoaks['topics']
    .fillna('Tidak Diketahui')
)

dim_topik = (
    df_hoaks[['topics']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_topik['id_topik'] = range(
    1,
    len(dim_topik) + 1
)

dim_topik = dim_topik[
    ['id_topik', 'topics']
]

print("dim_topik selesai")

dim_topik selesai


## 6.7 Pembuatan Dimensi Status Hoaks (dim_status_hoaks)

Dimensi ini digunakan untuk mengelompokkan status berita menjadi:

* Hoaks
* Non Hoaks
* Unknown

In [ ]:
df_hoaks['label_hoaks'] = (
    df_hoaks['label_hoaks']
    .fillna(-1)
)

dim_status_hoaks = pd.DataFrame({
    'id_status_hoaks': [-1, 0, 1],
    'status_hoaks': [
        'Unknown',
        'Non Hoaks',
        'Hoaks'
    ]
})

print("dim_status_hoaks selesai")

dim_status_hoaks selesai


In [ ]:
# cek kolom hoaks
print(df_hoaks.columns.tolist())

['author', 'clean_narasi', 'excerpt', 'id', 'isi', 'judul', 'kategori_asli', 'label_hoaks', 'link', 'main_image_url', 'narasi', 'politik', 'slug', 'sumber', 'tag', 'tanggal', 'text_clean', 'topics', 'view_count', 'tanggal_backup', 'id_hoaks', 'tahun', 'bulan', 'jumlah', 'panjang_isi', 'kategori']


In [ ]:
dim_tag.shape

(14479, 2)

# **7. BUILD FACT TABLE**

## 7.1 Integrasi Foreign Key ke Fact Table

Dilakukan proses join antara master data dengan seluruh dimension table untuk mendapatkan foreign key seperti:

* id_sumber
* id_waktu
* id_kategori
* id_topik
* id_tag

In [ ]:
fact_temp = df_hoaks.copy()

# join dim_berita
fact_temp = fact_temp.merge(
    dim_berita[['id_berita', 'judul']],
    on='judul',
    how='left'
)

# join dim_sumber
fact_temp = fact_temp.merge(
    dim_sumber,
    on='sumber',
    how='left'
)

# join dim_kategori
fact_temp = fact_temp.merge(
    dim_kategori,
    on='kategori',
    how='left'
)

# join dim_topik
fact_temp = fact_temp.merge(
    dim_topik,
    on='topics',
    how='left'
)

# join dim_waktu
fact_temp = fact_temp.merge(
    dim_waktu[['id_waktu', 'tanggal']],
    on='tanggal',
    how='left'
)

# status hoaks
fact_temp['id_status_hoaks'] = (
    pd.to_numeric(fact_temp['label_hoaks'], errors='coerce')
    .fillna(0)
    .astype(int)
)

# tag aman
fact_temp['tag'] = fact_temp['tag'].fillna('Tidak Diketahui')

In [ ]:
fact_rows = []

for _, row in fact_temp.iterrows():

    for tag in clean_tags(row['tag']):

        tag = tag.lower().strip()

        id_tag = tag_map.get(tag)

        if 'tidak diketahui' not in tag_map:
            tag_map['tidak diketahui'] = max(tag_map.values()) + 1

        fact_rows.append({
            'id_berita': row['id_berita'],
            'id_tag': id_tag,
            'id_waktu': row['id_waktu'],
            'id_sumber': row['id_sumber'],
            'id_topik': row['id_topik'],
            'id_kategori': row['id_kategori'],
            'id_status_hoaks': row['id_status_hoaks'],

            # measures
            'jumlah': 1,
            'panjang_isi': len(str(row['isi']))
        })

In [ ]:
fact_hoaks = pd.DataFrame(fact_rows)

fact_hoaks = fact_hoaks.drop_duplicates()

fact_hoaks = fact_hoaks.drop_duplicates(['id_berita','id_tag'])

fact_hoaks['id_fact_hoaks'] = range(1, len(fact_hoaks) + 1)

fact_hoaks = fact_hoaks[
    [
        'id_fact_hoaks',
        'id_berita',
        'id_tag',
        'id_waktu',
        'id_sumber',
        'id_topik',
        'id_kategori',
        'id_status_hoaks',
        'jumlah',
        'panjang_isi'
    ]
]

print("fact_hoaks selesai")

fact_hoaks selesai


# **8. EXPORT DATA WAREHOUSE KE FORMAT SQL**

## 8.1 Pembuatan Fungsi SQL Export

Dibuat fungsi:
* sql_safe() untuk membersihkan karakter khusus SQL
* mysql_dtype() untuk menentukan tipe data MySQL
* export_mysql_sql() untuk menghasilkan file SQL otomatis

In [ ]:
def sql_safe(value):

    if pd.isna(value):
        return 'NULL'

    value = str(value)

    # escape SQL
    value = value.replace("\\", "\\\\")
    value = value.replace("'", "''")
    value = value.replace('"', '\\"')

    # hapus newline
    value = value.replace("\n", " ")
    value = value.replace("\r", " ")

    return f"'{value}'"

In [ ]:
def mysql_dtype(col, dtype):

    col = col.lower()

    # surrogate key / foreign key
    if col.startswith('id_'):
        return 'INT'

    # tanggal
    if 'tanggal' in col:
        return 'DATETIME'

    # angka
    if 'jumlah' in col:
        return 'INT'

    if 'tahun' in col:
        return 'INT'

    if 'bulan' in col:
        return 'INT'

    if 'hari' in col:
        return 'INT'

    if 'quarter' in col:
        return 'INT'

    if 'panjang' in col:
        return 'INT'

    # default text
    return 'LONGTEXT'

def export_mysql_sql(
    df,
    table_name,
    file_name,
    create_table=True
):

    with open(file_name, 'w', encoding='utf-8') as f:

        # =========================
        # CREATE TABLE
        # =========================

        if create_table:

            # drop table
            f.write(
                f"\nDROP TABLE IF EXISTS `{table_name}`;\n"
            )

            # create table
            f.write(
                f"CREATE TABLE `{table_name}` (\n"
            )

            cols = []

            for col in df.columns:

                sql_type = mysql_dtype(
                    col,
                    df[col].dtype
                )

                cols.append(
                    f"`{col}` {sql_type}"
                )

            f.write(",\n".join(cols))

            f.write("\n);\n\n")

        # =========================
        # INSERT
        # =========================

        columns = ",".join(
            [f"`{c}`" for c in df.columns]
        )

        for _, row in df.iterrows():

            values = ",".join(
                [sql_safe(v) for v in row]
            )

            query = f"""
INSERT INTO `{table_name}` ({columns})
VALUES ({values});
"""

            f.write(query)

    print(f"{table_name} berhasil diexport")

In [ ]:
print(dim_kategori.head())

print(dim_status_hoaks.head())

print(fact_hoaks.head())

   id_kategori           kategori
0            1  Klarifikasi Hoaks
1            2    Tidak Diketahui
   id_status_hoaks status_hoaks
0               -1      Unknown
1                0    Non Hoaks
2                1        Hoaks
   id_fact_hoaks  id_berita  id_tag  id_waktu  id_sumber  id_topik  \
0              1          1    4264         1          1         1   
1              2          2    4264         2          1         1   
2              3          3    4264         3          1         1   
3              4          4    4264         4          1         1   
4              5          5    4264         5          1         1   

   id_kategori  id_status_hoaks  jumlah  panjang_isi  
0            1               -1       1          863  
1            1               -1       1         1029  
2            1               -1       1          701  
3            1               -1       1         1533  
4            1               -1       1          905  


## 8.2 Export Seluruh Dimension Table dan Fact Table

Seluruh tabel dimension dan fact diexport ke file SQL agar dapat diimport ke database MySQL.

In [ ]:
open('eis_mysql_final.sql', 'w').close()

export_mysql_sql(
    dim_berita,
    'dim_berita',
    'eis_mysql_final.sql'
)

export_mysql_sql(
    dim_sumber,
    'dim_sumber',
    'eis_mysql_final.sql'
)

export_mysql_sql(
    dim_waktu,
    'dim_waktu',
    'eis_mysql_final.sql'
)

export_mysql_sql(
    dim_kategori,
    'dim_kategori',
    'eis_mysql_final.sql'
)

export_mysql_sql(
    dim_status_hoaks,
    'dim_status_hoaks',
    'eis_mysql_final.sql'
)

export_mysql_sql(
    dim_tag,
    'dim_tag',
    'eis_mysql_final.sql'
)

export_mysql_sql(
    dim_topik,
    'dim_topik',
    'eis_mysql_final.sql'
)

export_mysql_sql(
    fact_hoaks,
    'fact_hoaks',
    'eis_mysql_final.sql'
)

print("MYSQL FINAL EXPORT BERHASIL")

dim_berita berhasil diexport
dim_sumber berhasil diexport
dim_waktu berhasil diexport
dim_kategori berhasil diexport
dim_status_hoaks berhasil diexport
dim_tag berhasil diexport
dim_topik berhasil diexport
fact_hoaks berhasil diexport
MYSQL FINAL EXPORT BERHASIL


In [ ]:
export_mysql_sql(dim_sumber, 'dim_sumber', 'dim_sumber.sql')

export_mysql_sql(dim_berita, 'dim_berita', 'dim_berita.sql')

export_mysql_sql(dim_waktu, 'dim_waktu', 'dim_waktu.sql')

export_mysql_sql(dim_tag, 'dim_tag', 'dim_tag.sql')

export_mysql_sql(dim_kategori, 'dim_kategori', 'dim_kategori.sql')

export_mysql_sql(dim_topik, 'dim_topik', 'dim_topik.sql')

export_mysql_sql(dim_status_hoaks, 'dim_status_hoaks', 'dim_status_hoaks.sql')

export_mysql_sql(fact_hoaks, 'fact_hoaks', 'fact_hoaks.sql')

dim_sumber berhasil diexport
dim_berita berhasil diexport
dim_waktu berhasil diexport
dim_tag berhasil diexport
dim_kategori berhasil diexport
dim_topik berhasil diexport
dim_status_hoaks berhasil diexport
fact_hoaks berhasil diexport


## 8.3 Export Fact Table Berdasarkan Tahun

Fact table dipisahkan berdasarkan tahun dan diexport menjadi beberapa file SQL untuk mempermudah manajemen data dalam skala besar.

In [ ]:
fact_export = fact_hoaks.merge(

    dim_waktu[
        ['id_waktu', 'tahun']
    ],

    on='id_waktu',

    how='left'
)

print(fact_export.head())

   id_fact_hoaks  id_berita  id_tag  id_waktu  id_sumber  id_topik  \
0              1          1    4264         1          1         1   
1              2          2    4264         2          1         1   
2              3          3    4264         3          1         1   
3              4          4    4264         4          1         1   
4              5          5    4264         5          1         1   

   id_kategori  id_status_hoaks  jumlah  panjang_isi   tahun  
0            1               -1       1          863  2026.0  
1            1               -1       1         1029  2026.0  
2            1               -1       1          701  2026.0  
3            1               -1       1         1533  2026.0  
4            1               -1       1          905  2026.0  


In [ ]:
years = sorted(
    fact_export['tahun']
    .dropna()
    .unique()
)

first = True

for y in years:

    temp = fact_export[
        fact_export['tahun'] == y
    ]

    filename = f'fact_{int(y)}.sql'

    export_mysql_sql(

        temp,

        'fact_hoaks',

        filename,

        create_table=first

    )

    first = False

    print(filename, "berhasil dibuat")

fact_hoaks berhasil diexport
fact_2015.sql berhasil dibuat
fact_hoaks berhasil diexport
fact_2016.sql berhasil dibuat
fact_hoaks berhasil diexport
fact_2017.sql berhasil dibuat
fact_hoaks berhasil diexport
fact_2018.sql berhasil dibuat
fact_hoaks berhasil diexport
fact_2019.sql berhasil dibuat
fact_hoaks berhasil diexport
fact_2020.sql berhasil dibuat
fact_hoaks berhasil diexport
fact_2021.sql berhasil dibuat
fact_hoaks berhasil diexport
fact_2022.sql berhasil dibuat
fact_hoaks berhasil diexport
fact_2023.sql berhasil dibuat
fact_hoaks berhasil diexport
fact_2024.sql berhasil dibuat
fact_hoaks berhasil diexport
fact_2025.sql berhasil dibuat
fact_hoaks berhasil diexport
fact_2026.sql berhasil dibuat


# **9. VERIFIKASI AKHIR DATA WAREHOUSE**

Verifikasi akhir dilakukan untuk memastikan fact table berhasil dibentuk tanpa missing value, duplicate data, maupun relasi yang tidak konsisten.

## 9.1 Cek Missing Value Fact Table

In [ ]:
print("Missing Value Fact Table")
print(fact_hoaks.isnull().sum())

Missing Value Fact Table
id_fact_hoaks      0
id_berita          0
id_tag             0
id_waktu           0
id_sumber          0
id_topik           0
id_kategori        0
id_status_hoaks    0
jumlah             0
panjang_isi        0
dtype: int64


## 9.2 Cek Jumlah Record Fact Table

In [ ]:
print("Jumlah record fact table:")
print(len(fact_hoaks))

Jumlah record fact table:
147522


## 9.3 Cek Duplicate Relasi Berita dan Tag

In [ ]:
duplicate_fact = fact_hoaks.duplicated(
    subset=['id_berita', 'id_tag']
).sum()

print("Jumlah duplicate fact:")
print(duplicate_fact)

Jumlah duplicate fact:
0


## 9.4 Cek Keunikan Relasi Berita dan Tag

In [ ]:
print(
    len(fact_hoaks),
    fact_hoaks[
        ['id_berita', 'id_tag']
    ].drop_duplicates().shape[0]
)

147522 147522


## 9.5 Distribusi Data Berdasarkan Sumber

In [ ]:
df_hoaks['sumber'].value_counts()

,count
sumber,
Komdigi,16258
TurnBackHoax,10381
CNN,9630
Tempo,6592
Kompas,4750
